# Elliptic Bitcoin Transactions: Serving Export

Notebook 7 of 7. It packages everything the live API needs into one folder: the test-period transactions with their features, edges and precomputed scores, the GraphSAGE weights, the hybrid LightGBM model, serving metadata and the chart data for the model page. The folder is uploaded to a private Hugging Face model repository, never committed to GitHub.

## Setup

Imports and helpers. `SMOKE` changes nothing here except the label printed, since the export is fast; it is off in this version.

In [1]:
import importlib.metadata as metadata
import json
import platform
import shutil
import time
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_curve

SMOKE = False
TRAIN_LAST = 34
PYG_VERSION = "2.8.0.post1"
MODELS = ["hybrid", "graphsage", "raw_eng", "rf"]
STACKING = ["gnn_logit", "gnn_in_mean", "gnn_in_max", "gnn_out_mean", "gnn_out_max"]
INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working")
ART = OUT / "artifacts"
ART.mkdir(parents=True, exist_ok=True)
START = time.time()


def find_file(name):
    hits = sorted(INPUT.rglob(name))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


def elapsed():
    return f"elapsed {(time.time() - START) / 60:.1f} minutes"


print(f"smoke={SMOKE}")

smoke=False


**Conclusion.** The smoke switch is off, so the bundle is built from the final upstream outputs.

## The node store

Aim: keep only the test period (steps 35 to 49), so every transaction the console shows is out of sample, and store its raw features, engineered features, stacking features, the four models' scores and its edges as compact float32 and int32 arrays in `store.npz`.

In [2]:
feat = pd.read_parquet(find_file("features.parquet"))
edges = pd.read_parquet(find_file("edges.parquet"))
meta = json.loads(find_file("feature_meta.json").read_text(encoding="utf-8"))
gnn = pd.read_parquet(find_file("gnn_scores.parquet"))
pred = pd.read_parquet(find_file("test_pred.parquet"))
metrics = json.loads(find_file("metrics.json").read_text(encoding="utf-8"))
random_metrics = json.loads(find_file("metrics_random.json").read_text(encoding="utf-8"))
RAW, ENG, ATTRS = meta["raw"], meta["engineered"], meta["attrs"]
nodes = feat[feat.time_step > TRAIN_LAST].reset_index(drop=True)
ids = nodes.txId.to_numpy()
scores = pred.set_index("txId").loc[ids]
index = pd.Series(np.arange(len(ids)), index=ids)
inside = edges.txId1.isin(index.index) & edges.txId2.isin(index.index)
store = {
    "tx_id": ids.astype(np.int64),
    "time_step": nodes.time_step.to_numpy().astype(np.int16),
    "label": nodes.label.to_numpy().astype(np.int8),
    "raw": nodes[RAW].to_numpy(np.float32),
    "eng": nodes[ENG].to_numpy(np.float32),
    "stack": gnn.set_index("txId").loc[ids, STACKING].to_numpy(np.float32),
    **{f"score_{m}": scores[m].to_numpy(np.float32) for m in MODELS},
    "src": index.loc[edges.txId1[inside]].to_numpy().astype(np.int32),
    "dst": index.loc[edges.txId2[inside]].to_numpy().astype(np.int32),
}
np.savez(ART / "store.npz", **store)
print(f"store: {len(ids):,} transactions, {len(store['src']):,} edges, steps {store['time_step'].min()} to {store['time_step'].max()}")
print(f"labelled {(store['label'] >= 0).sum():,}, illicit {(store['label'] == 1).sum():,}")
print(f"in-memory size of the arrays: {sum(v.nbytes for v in store.values()) / 1e6:.1f} MB")

store: 67,504 transactions, 77,512 edges, steps 35 to 49
labelled 16,670, illicit 1,083
in-memory size of the arrays: 96.1 MB


**Conclusion.** The store holds all 67,504 test-period transactions (16,670 labelled, 1,083 illicit) and the 77,512 edges among them, steps 35 to 49 only, so every score the console shows is out of sample. Its arrays take 96.1 MB in memory, the largest part of the app's footprint on Render's 512 MB free tier.

## Check the hybrid model against the store

Aim: confirm that the saved LightGBM model, fed the stored float32 features in the stored order, reproduces notebook 5's test scores, which is the same computation the API performs for an existing transaction.

In [3]:
booster = lgb.Booster(model_file=str(find_file("lgbm_hybrid.txt")))
recomputed = booster.predict(np.hstack([store["raw"], store["eng"], store["stack"]]).astype(np.float64))
print(f"booster feature order matches raw + engineered + stacking: {booster.feature_name() == RAW + ENG + STACKING}")
print(f"hybrid scores recomputed from the store, max abs difference: {np.abs(recomputed - store['score_hybrid']).max():.2e}")

booster feature order matches raw + engineered + stacking: True
hybrid scores recomputed from the store, max abs difference: 2.98e-08


**Conclusion.** The saved hybrid booster expects its features in exactly the store's order (raw, engineered, stacking), and fed the stored float32 features it reproduces notebook 5's test scores to within 3e-08, the rounding of storing scores as float32. The API therefore computes the same scores the evaluation reported.

## Serving metadata

Aim: write `feature_meta.json` for the app: feature orders, the headline model and every threshold, the test steps, and for the new-transaction form the eight raw features the hybrid model relies on most (by gain) plus the test-period median of every raw feature. The form starts from medians rather than a real transaction so that no full raw row ever leaves the API.

In [4]:
gain = pd.Series(booster.feature_importance(importance_type="gain"), index=booster.feature_name())
form_features = list(gain[RAW].sort_values(ascending=False).index[:8])
serving = {
    "raw": RAW, "attrs": ATTRS, "structural": meta["structural"], "engineered": ENG, "stacking": STACKING,
    "hybrid": RAW + ENG + STACKING, "gnn_inputs": RAW + meta["structural"],
    "headline": metrics["headline"],
    "thresholds": {m: float(metrics["thresholds"][m]) for m in MODELS},
    "steps": sorted(int(s) for s in np.unique(store["time_step"])),
    "form_features": form_features,
    "medians": {name: float(np.median(store["raw"][:, i])) for i, name in enumerate(RAW)},
    "model_names": metrics["model_names"],
    "counts": {"test_nodes": int(len(ids)), "test_labelled": int((store["label"] >= 0).sum()), "test_illicit": int((store["label"] == 1).sum())},
}
(ART / "feature_meta.json").write_text(json.dumps(serving, indent=2, allow_nan=False), encoding="utf-8")
print(f"headline {serving['headline']}, thresholds {({k: round(v, 4) for k, v in serving['thresholds'].items()})}")
print(f"form features: {form_features}")

headline hybrid, thresholds {'hybrid': 0.4074, 'graphsage': 0.647, 'raw_eng': 0.1737, 'rf': 0.2955}
form features: ['local_53', 'local_41', 'local_55', 'local_14', 'local_90', 'agg_39', 'agg_70', 'local_2']


**Conclusion.** The serving metadata names the hybrid as the headline model with its out-of-fold threshold of 0.4074, and carries the other models' thresholds for the console's comparison bars. The new-transaction form exposes the eight raw features the hybrid relies on most by gain, led by local_53, local_41 and local_55, and starts every other raw feature at its test-period median, so no real transaction's full row is ever exposed.

## Chart data

Aim: write `charts.json` for the model page: test-period ROC and precision-recall curves thinned to about 200 points per model, per-step F1, false negatives at own and matched thresholds, the ablation, the temporal against random contrast and the benchmark table.

In [5]:
labelled = pred[pred.label >= 0]
y_true = labelled.label.to_numpy().astype(int)


def thin(x, y, points=200):
    keep = np.unique(np.linspace(0, len(x) - 1, points).round().astype(int))
    return [round(float(v), 5) for v in x[keep]], [round(float(v), 5) for v in y[keep]]


roc, pr = {}, {}
for m in MODELS:
    p = labelled[m].to_numpy().astype(float)
    fpr, tpr, _ = roc_curve(y_true, p)
    x, y = thin(fpr, tpr)
    roc[m] = {"x": x, "y": y, "area": float(metrics["test"][m]["auc"])}
    precision, recall, _ = precision_recall_curve(y_true, p)
    x, y = thin(recall, precision)
    pr[m] = {"x": x, "y": y, "area": float(metrics["test"][m]["pr_auc"])}
charts = {
    "model_names": metrics["model_names"], "headline": metrics["headline"], "roc": roc, "pr": pr,
    "per_step": {"steps": metrics["per_step"]["steps"], "illicit": metrics["per_step"]["illicit"], "f1": metrics["per_step"]["f1"]},
    "fn": {m: {"own": int(metrics["test"][m]["fn"]), "matched": int(metrics["matched"][m]["fn"]) if m in metrics["matched"] else None} for m in MODELS},
    "ablation": metrics["ablation"], "contrast": random_metrics["contrast"], "benchmarks": metrics["benchmarks"], "counts": metrics["counts"],
}
(ART / "charts.json").write_text(json.dumps(charts, allow_nan=False), encoding="utf-8")
print(f"charts.json: {(ART / 'charts.json').stat().st_size / 1e3:.0f} KB, curves for {len(roc)} models")

charts.json: 33 KB, curves for 4 models


**Conclusion.** The model page's chart data takes 33 KB: ROC and precision-recall curves for the four models thinned to about 200 points each, per-step F1, false negatives, the ablation, the temporal against random contrast and the benchmark table.

## Models, metrics and versions

Aim: copy the GraphSAGE weights, the hybrid booster and both metric files into the bundle, record library versions so the app pins the same ones, and list the bundle's files and sizes.

In [6]:
shutil.copy(find_file("graphsage.npz"), ART / "graphsage.npz")
shutil.copy(find_file("lgbm_hybrid.txt"), ART / "lgbm.txt")
shutil.copy(find_file("metrics.json"), ART / "metrics.json")
shutil.copy(find_file("metrics_random.json"), ART / "metrics_random.json")
versions = {"python": platform.python_version(), **{name: metadata.version(dist) for name, dist in [("numpy", "numpy"), ("scipy", "scipy"), ("networkx", "networkx"), ("lightgbm", "lightgbm"), ("pandas", "pandas"), ("sklearn", "scikit-learn"), ("torch", "torch")]}, "torch_geometric": PYG_VERSION}
(ART / "versions.json").write_text(json.dumps(versions, indent=2), encoding="utf-8")
print(json.dumps(versions, indent=2))
total = 0
for path in sorted(ART.iterdir()):
    total += path.stat().st_size
    print(f"{path.name}: {path.stat().st_size / 1e6:.2f} MB")
print(f"bundle total {total / 1e6:.1f} MB")
print(elapsed())

{
  "python": "3.13.15",
  "numpy": "2.1.3",
  "scipy": "1.16.3",
  "networkx": "3.6.1",
  "lightgbm": "4.6.0",
  "pandas": "2.3.3",
  "sklearn": "1.6.1",
  "torch": "2.11.0+cpu",
  "torch_geometric": "2.8.0.post1"
}
charts.json: 0.03 MB
feature_meta.json: 0.02 MB
graphsage.npz: 3.38 MB
lgbm.txt: 1.45 MB
metrics.json: 0.02 MB
metrics_random.json: 0.01 MB
store.npz: 96.14 MB
versions.json: 0.00 MB
bundle total 101.1 MB
elapsed 0.1 minutes


**Conclusion.** The bundle is complete: eight files totalling 101.1 MB, dominated by the 96.1 MB store, with the 3.4 MB float64 GraphSAGE weights and the 1.5 MB booster. versions.json records the Kaggle library versions (numpy 2.1.3, scipy 1.16.3, networkx 3.6.1, lightgbm 4.6.0) that the serving container pins, so the app computes features and scores with the same code paths.